<a href="https://colab.research.google.com/github/swaggincoffee-bit/Tesi/blob/main/01_Preparazione_dati.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 1 — Mount Google Drive
# ══════════════════════════════════════════════════════════════════════════════

from google.colab import drive
drive.mount("/content/drive")

import os

# ── Percorsi sorgente (My Drive\Contatori\CSV\) ───────────────────────────────
BASE     = "/content/drive/MyDrive/Contatori/CSV"
PATH_SAP  = os.path.join(BASE, "Parco Contatori_G4-G6_20251231_SAP.csv")
PATH_BEAM = os.path.join(BASE, "TABELLA_CONTATORE_BEAM.csv")
PATH_LETT = os.path.join(BASE, "TABELLA_LETTURE_BEAM.csv")

# ── Cartella di output (creata automaticamente se non esiste) ─────────────────
OUTPUT_DIR = "/content/drive/MyDrive/Contatori/OUTPUT/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verifica esistenza file
for label, path in [("SAP", PATH_SAP), ("BEAM", PATH_BEAM), ("LETTURE", PATH_LETT)]:
    exists = os.path.exists(path)
    print(f"  {'✅' if exists else '❌'} {label}: {path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  ✅ SAP: /content/drive/MyDrive/Contatori/CSV/Parco Contatori_G4-G6_20251231_SAP.csv
  ✅ BEAM: /content/drive/MyDrive/Contatori/CSV/TABELLA_CONTATORE_BEAM.csv
  ✅ LETTURE: /content/drive/MyDrive/Contatori/CSV/TABELLA_LETTURE_BEAM.csv


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 2 — Installazione librerie mancanti in Colab
# ══════════════════════════════════════════════════════════════════════════════

# statsmodels è già presente in Colab; le altre librerie standard pure.
# Se una libreria mancasse, decommentare la riga corrispondente.

!pip install -q statsmodels --upgrade



In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 3 — Import e configurazione grafica
# ══════════════════════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import datetime, timedelta

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, roc_auc_score, roc_curve,
    ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance

plt.rcParams.update({
    "figure.dpi": 120,
    "figure.figsize": (12, 5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "sans-serif",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})
PALETTE = sns.color_palette("muted")
sns.set_style("whitegrid")

print("✅ Librerie importate correttamente.")


✅ Librerie importate correttamente.


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 4 — Caricamento dati grezzi
# ══════════════════════════════════════════════════════════════════════════════

def load_csv(path, label):
    """Carica un CSV provando diversi encoding comuni."""
    for enc in ["utf-8", "latin-1", "cp1252", "utf-8-sig"]:
        try:
            df = pd.read_csv(path, sep=";", encoding=enc, dtype=str, low_memory=False)
            df.columns = df.columns.str.strip()
            print(f"  ✅ {label}: {df.shape[0]:,} righe × {df.shape[1]} colonne  (encoding: {enc})")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Impossibile leggere {path} con nessun encoding provato.")

print("── Caricamento file ──")
df_sap  = load_csv(PATH_SAP,  "SAP")
df_beam = load_csv(PATH_BEAM, "BEAM")
df_lett = load_csv(PATH_LETT, "LETTURE")

# Preview colonne
print("\n── Colonne SAP:")
print(df_sap.columns.tolist())
print("\n── Colonne BEAM:")
print(df_beam.columns.tolist())
print("\n── Colonne LETTURE:")
print(df_lett.columns.tolist())


── Caricamento file ──
  ✅ SAP: 712,571 righe × 13 colonne  (encoding: latin-1)
  ✅ BEAM: 761,967 righe × 7 colonne  (encoding: utf-8)
  ✅ LETTURE: 1,048,001 righe × 7 colonne  (encoding: utf-8)

── Colonne SAP:
['04IND - Provincia (cod)', '04IND - Comune (cod)', '04IND - CAP (cod)', '03IMPS - STec - Accessibile (cod)', '03IMPS - PDR/POD - Esterno (cod)', '03IMPS - Stato telegestione (cod)', '03IMPS - Operandi - Cons.Anno PDR (desc)', '11MDEF - Installazione(dta)', '11MDEF - Costruttore (cod)', '11MDEF - Anno Costruzione (cod)', '11MDEF - Materiale (cod)', '11MDEF - Numero Serie (cod)', 'MEAS - PDR (conteggio)']

── Colonne BEAM:
['MATRICOLA CONTATORE', 'PDR', 'MODELLO CONTATORE', 'VERSIONE FIRMWARE', 'Tecnologia di comunicazione', 'Data ultima comunicazione', 'Data ultima misura']

── Colonne LETTURE:
['PDR', 'Matricola Contatore', 'Timestamp comunicazione', 'Data lettura', 'Stato lettura', 'Diagnostica Contatore', 'Unnamed: 6']


In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 5 — Ispezione e mapping colonne
# ══════════════════════════════════════════════════════════════════════════════

# Stampa prime 3 righe di ogni dataset per verifica visiva
print("═" * 60)
print("SAP — prime 3 righe")
display(df_sap.head(3))

print("═" * 60)
print("BEAM — prime 3 righe")
display(df_beam.head(3))

print("═" * 60)
print("LETTURE — prime 3 righe")
display(df_lett.head(3))

════════════════════════════════════════════════════════════
SAP — prime 3 righe


,04IND - Provincia (cod),04IND - Comune (cod),04IND - CAP (cod),03IMPS - STec - Accessibile (cod),03IMPS - PDR/POD - Esterno (cod),03IMPS - Stato telegestione (cod),03IMPS - Operandi - Cons.Anno PDR (desc),11MDEF - Installazione(dta),11MDEF - Costruttore (cod),11MDEF - Anno Costruzione (cod),11MDEF - Materiale (cod),11MDEF - Numero Serie (cod),MEAS - PDR (conteggio)
0,Genova,Avegno,16036,ACCESSIBILE,03270006541254,GT,705,30.09.2022,METERSIT,2022,000000000001072798,MTSB030F06147670,1
1,Genova,Avegno,16036,ACCESSIBILE,03270006545605,GT,1110,07.10.2022,METERSIT,2022,000000000001072798,MTSB030F06147565,1
2,Genova,Avegno,16036,ACCESSIBILE,03270006547423,GT,615,25.11.2022,METERSIT,2022,000000000001072798,MTSB030F06147637,1


════════════════════════════════════════════════════════════
BEAM — prime 3 righe


,MATRICOLA CONTATORE,PDR,MODELLO CONTATORE,VERSIONE FIRMWARE,Tecnologia di comunicazione,Data ultima comunicazione,Data ultima misura
0,SMGR034016059502,15441000054940,RSE/2001 LA G4,229,RF,04-FEB-26 05.35.49.000000000 PM,03-FEB-26 12.00.00.000000000 AM
1,SMGR034016679960,15441000054953,RSE/2001 LA G4,230,RF,04-FEB-26 07.26.34.000000000 AM,03-FEB-26 12.00.00.000000000 AM
2,SMGR034017243004,15441000054559,RSE/2001 LA G4,212,RF,04-FEB-26 07.38.00.000000000 AM,04-FEB-26 12.00.00.000000000 AM


════════════════════════════════════════════════════════════
LETTURE — prime 3 righe


,PDR,Matricola Contatore,Timestamp comunicazione,Data lettura,Stato lettura,Diagnostica Contatore,Unnamed: 6
0,15441000040111,MTSB036406264806,17-DEC-25 10.52.56.156000000 AM,16-DEC-25 12.00.00.000000000 AM,ELA,0000000000000000,NaN
1,15441000056279,MTSB036430056190,17-DEC-25 09.43.51.718000000 AM,16-DEC-25 12.00.00.000000000 AM,ELA,0000000000000000,NaN
2,03270015229272,MTSB036400928751,17-DEC-25 12.54.15.833000000 PM,16-DEC-25 12.00.00.000000000 AM,ELA,0000000000000000,NaN


In [10]:

# ══════════════════════════════════════════════════════════════════════════════
# CELLA 6 — Mapping nomi colonne
# (Aggiorna questi valori se i nomi nel tuo file differiscono dallo snippet)
# ══════════════════════════════════════════════════════════════════════════════

# ── SAP ──────────────────────────────────────────────────────────────────────
COL_PROV        = "04IND - Provincia (cod)"
COL_COMUNE      = "04IND - Comune (cod)"
COL_CAP         = "04IND - CAP (cod)"
COL_ACCESSIBILE = "03IMPS - STec - Accessibile (cod)"
COL_TELEGESTIONE= "03IMPS - Stato telegestione (cod)"
COL_CONSUMO     = "03IMPS - Operandi - Cons.Anno PDR (desc)"
COL_DATA_INST   = "11MDEF - Installazione(dta)"
COL_COSTRUTTORE = "11MDEF - Costruttore (cod)"
COL_ANNO_COSTR  = "11MDEF - Anno Costruzione (cod)"
COL_MATERIALE   = "11MDEF - Materiale (cod)"
KEY_SAP         = "11MDEF - Numero Serie (cod)"   # ← matricola in SAP

# ── BEAM ─────────────────────────────────────────────────────────────────────
KEY_BEAM     = "MATRICOLA CONTATORE"
COL_MODELLO  = "MODELLO CONTATORE"
COL_FIRMWARE = "VERSIONE FIRMWARE"
COL_TECN_COM = "Tecnologia di comunicazione"
COL_ULT_COM  = "Data ultima comunicazione"
COL_ULT_MIS  = "Data ultima misura"

# ── LETTURE ──────────────────────────────────────────────────────────────────
KEY_LETT         = "Matricola Contatore"
COL_DATA_LETT    = "Data lettura"
COL_STATO_LETT   = "Stato lettura"
COL_DIAGNOSTICA  = "Diagnostica Contatore"

# Verifica che le colonne chiave esistano
print("Verifica colonne chiave:")
checks = [
    (df_sap,  KEY_SAP,  "SAP — matricola"),
    (df_beam, KEY_BEAM, "BEAM — matricola"),
    (df_lett, KEY_LETT, "LETTURE — matricola"),
    (df_lett, COL_DATA_LETT, "LETTURE — data lettura"),
    (df_sap,  COL_PROV, "SAP — provincia"),
]
for df, col, label in checks:
    ok = col in df.columns
    print(f"  {'✅' if ok else '❌  ← DA CORREGGERE'} {label}: '{col}'")



Verifica colonne chiave:
  ✅ SAP — matricola: '11MDEF - Numero Serie (cod)'
  ✅ BEAM — matricola: 'MATRICOLA CONTATORE'
  ✅ LETTURE — matricola: 'Matricola Contatore'
  ✅ LETTURE — data lettura: 'Data lettura'
  ✅ SAP — provincia: '04IND - Provincia (cod)'


In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 7 — Filtraggio Reggio Emilia
# ══════════════════════════════════════════════════════════════════════════════

print("\nValori unici colonna provincia (SAP):")
print(df_sap[COL_PROV].value_counts().to_string())

# ⚠️ Verifica l'output sopra e aggiorna PROV_TARGET se necessario
PROV_TARGET = "Reggio nell'Emilia"   # ← adatta al valore esatto presente

df_sap_re = df_sap[
    df_sap[COL_PROV].str.contains(PROV_TARGET, case=False, na=False)
].copy()

print(f"\nContatori SAP — provincia RE: {len(df_sap_re):,}")
if len(df_sap_re) == 0:
    print("⚠️  Nessun contatore trovato. Controlla PROV_TARGET.")


Valori unici colonna provincia (SAP):
04IND - Provincia (cod)
Genova                299192
Reggio nell'Emilia    228465
Parma                 159463
Savona                 15934
Piacenza                9517

Contatori SAP — provincia RE: 228,465


In [21]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 8 — Merge anagrafica e costruzione parco contatori RE completo
# ══════════════════════════════════════════════════════════════════════════════

# ── Normalizzazione chiavi ────────────────────────────────────────────────────
# Stessa stringa → stesso formato in tutti e tre i dataset
df_sap_re        = df_sap_re.copy()
df_sap_re["_key"] = df_sap_re[KEY_SAP].str.strip().str.upper()
df_beam["_key"]   = df_beam[KEY_BEAM].str.strip().str.upper()
df_lett["_key"]   = df_lett[KEY_LETT].str.strip().str.upper()

print(f"Contatori unici SAP-RE  : {df_sap_re['_key'].nunique():,}")
print(f"Contatori unici BEAM    : {df_beam['_key'].nunique():,}")
print(f"Contatori unici LETTURE : {df_lett['_key'].nunique():,}")

# ── Merge SAP ↔ BEAM (inner) ──────────────────────────────────────────────────
# Teniamo solo i contatori presenti in entrambe le anagrafiche.
# Un contatore senza dati SAP o senza dati BEAM ha anagrafica incompleta
# e non può essere analizzato → scartato.
df_anagrafica = df_sap_re.merge(
    df_beam, on="_key", how="inner", suffixes=("_sap", "_beam")
)
print(f"\nDopo merge SAP↔BEAM (inner) : {len(df_anagrafica):,} contatori")
print(f"Scartati per anagrafica incompleta: {df_sap_re['_key'].nunique() - len(df_anagrafica):,}")

# ── Subset letture: solo quelle relative al parco RE ─────────────────────────
# Non filtriamo df_anagrafica sulle letture.
# Il parco contatori SAP-BEAM è l'universo di riferimento: tutti i contatori
# fisicamente installati in provincia RE. Le letture sono ciò che quei
# contatori hanno *eventualmente* prodotto.
# Un contatore senza letture non viene eliminato: la sua assenza totale di
# comunicazioni è il segnale più forte di guasto ed è la nostra variabile
# target al suo valore massimo (silente su tutti gli orizzonti).
df_lett_re = df_lett[
    df_lett["_key"].isin(df_anagrafica["_key"])
].copy()

# ── Diagnostica: contatori con e senza letture ───────────────────────────────
matricole_parco    = set(df_anagrafica["_key"])
matricole_con_lett = set(df_lett_re["_key"])
matricole_senza_lett = matricole_parco - matricole_con_lett

n_tot    = len(matricole_parco)
n_con    = len(matricole_con_lett)
n_senza  = len(matricole_senza_lett)

print(f"\nParco contatori RE totale     : {n_tot:,}")
print(f"  ├─ Con almeno una lettura   : {n_con:,}  ({n_con/n_tot*100:.1f}%)")
print(f"  └─ Senza alcuna lettura     : {n_senza:,}  ({n_senza/n_tot*100:.1f}%)  → target=1 su tutti gli orizzonti")
print(f"\nLetture filtrate (RE)         : {len(df_lett_re):,} righe")

# Conserva il set per usarlo nella cella 10 (costruzione target)
# I contatori senza letture riceveranno direttamente target=1 su tutti
# gli orizzonti senza passare per ha_settimana_senza_lettura()


Contatori unici SAP-RE  : 228,465
Contatori unici BEAM    : 761,940
Contatori unici LETTURE : 523,177

Dopo merge SAP↔BEAM (inner) : 226,707 contatori
Scartati per anagrafica incompleta: 1,758

Parco contatori RE totale     : 226,702
  ├─ Con almeno una lettura   : 157,746  (69.6%)
  └─ Senza alcuna lettura     : 68,956  (30.4%)  → target=1 su tutti gli orizzonti

Letture filtrate (RE)         : 298,745 righe


In [22]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 8b — Print di esempio
# ══════════════════════════════════════════════════════════════════════════════

print("── df_anagrafica", df_anagrafica.shape)
display(df_anagrafica.head(3))

print("\n── df_lett_re", df_lett_re.shape)
display(df_lett_re.head(3))

# Letture di un singolo contatore
esempio_key = df_lett_re["_key"].value_counts().index[0]
print(f"\n── Letture contatore {esempio_key} ({df_lett_re['_key'].value_counts().iloc[0]} righe)")
display(df_lett_re[df_lett_re["_key"] == esempio_key].sort_values("Data lettura"))

# Un contatore silente
esempio_silente = list(matricole_senza_lett)[0]
print(f"\n── Contatore silente {esempio_silente} (0 letture)")
display(df_anagrafica[df_anagrafica["_key"] == esempio_silente])

── df_anagrafica (226707, 21)


,04IND - Provincia (cod),04IND - Comune (cod),04IND - CAP (cod),03IMPS - STec - Accessibile (cod),03IMPS - PDR/POD - Esterno (cod),03IMPS - Stato telegestione (cod),03IMPS - Operandi - Cons.Anno PDR (desc),11MDEF - Installazione(dta),11MDEF - Costruttore (cod),11MDEF - Anno Costruzione (cod),...,11MDEF - Numero Serie (cod),MEAS - PDR (conteggio),_key,MATRICOLA CONTATORE,PDR,MODELLO CONTATORE,VERSIONE FIRMWARE,Tecnologia di comunicazione,Data ultima comunicazione,Data ultima misura
0,Reggio nell'Emilia,Albinea,42020,ACCESSIBILE,15441000000750,GT,916,17.12.2024,METERSIT,2024,...,MTSB030F07843782,1,MTSB030F07843782,MTSB030F07843782,15441000000750,G4_NBIOT,O732,NBIOT,31-JAN-26 02.58.30.388000000 PM,30-JAN-26 12.00.00.000000000 AM
1,Reggio nell'Emilia,Albinea,42020,ACCESSIBILE,15441000001565,GT,2182,22.10.2024,METERSIT,2024,...,MTSB030F10231825,1,MTSB030F10231825,MTSB030F10231825,15441000001565,G6_NBIOT,O732,NBIOT,04-FEB-26 06.57.59.313000000 AM,03-FEB-26 12.00.00.000000000 AM
2,Reggio nell'Emilia,Albinea,42020,ACCESSIBILE,15441000002472,GT,156,11.08.2017,METERSIT,2017,...,MTSB036400981217,1,MTSB036400981217,MTSB036400981217,15441000002472,DOMUS NEXT G4,O400A396,GPRS,05-FEB-26 11.50.19.697000000 AM,03-FEB-26 12.00.00.000000000 AM



── df_lett_re (298745, 8)


,PDR,Matricola Contatore,Timestamp comunicazione,Data lettura,Stato lettura,Diagnostica Contatore,Unnamed: 6,_key
0,15441000040111,MTSB036406264806,17-DEC-25 10.52.56.156000000 AM,16-DEC-25 12.00.00.000000000 AM,ELA,0000000000000000,NaN,MTSB036406264806
3,15441000247193,MTSB036401961029,17-DEC-25 11.23.44.841000000 AM,16-DEC-25 12.00.00.000000000 AM,ELA,0000000000000000,NaN,MTSB036401961029
5,15441000257032,MTSB036502090279,16-DEC-25 10.11.07.000000000 AM,13-DEC-25 12.00.00.000000000 AM,ERR,0000000000000000,NaN,MTSB036502090279



── Letture contatore MTSB036506270630 (22 righe)


,PDR,Matricola Contatore,Timestamp comunicazione,Data lettura,Stato lettura,Diagnostica Contatore,Unnamed: 6,_key
373255,16360000002276,MTSB036506270630,16-DEC-25 03.52.01.000000000 AM,12-DEC-25 12.00.00.000000000 AM,ERR,0000100000000000,NaN,MTSB036506270630
775399,16360000002276,MTSB036506270630,16-DEC-25 07.52.14.000000000 AM,12-DEC-25 12.00.00.000000000 AM,ERR,0000100000000000,NaN,MTSB036506270630
766609,16360000002276,MTSB036506270630,16-DEC-25 06.32.53.000000000 AM,12-DEC-25 12.00.00.000000000 AM,ERR,0000100000000000,NaN,MTSB036506270630
716613,16360000002276,MTSB036506270630,16-DEC-25 02.13.06.000000000 PM,12-DEC-25 12.00.00.000000000 AM,ERR,0000100000000000,NaN,MTSB036506270630
716612,16360000002276,MTSB036506270630,16-DEC-25 01.12.53.000000000 PM,12-DEC-25 12.00.00.000000000 AM,ERR,0000100000000000,NaN,MTSB036506270630
716611,16360000002276,MTSB036506270630,16-DEC-25 10.33.06.000000000 AM,12-DEC-25 12.00.00.000000000 AM,ERR,0000100000000000,NaN,MTSB036506270630
716610,16360000002276,MTSB036506270630,16-DEC-25 09.12.40.000000000 AM,12-DEC-25 12.00.00.000000000 AM,ERR,0000100000000000,NaN,MTSB036506270630
716609,16360000002276,MTSB036506270630,16-DEC-25 08.52.13.000000000 AM,12-DEC-25 12.00.00.000000000 AM,ERR,0000100000000000,NaN,MTSB036506270630
716607,16360000002276,MTSB036506270630,16-DEC-25 08.12.40.000000000 AM,12-DEC-25 12.00.00.000000000 AM,ERR,0000100000000000,NaN,MTSB036506270630
671845,16360000002276,MTSB036506270630,16-DEC-25 12.33.19.000000000 PM,12-DEC-25 12.00.00.000000000 AM,ERR,0000100000000000,NaN,MTSB036506270630



── Contatore silente MTSB030F07844927 (0 letture)


,04IND - Provincia (cod),04IND - Comune (cod),04IND - CAP (cod),03IMPS - STec - Accessibile (cod),03IMPS - PDR/POD - Esterno (cod),03IMPS - Stato telegestione (cod),03IMPS - Operandi - Cons.Anno PDR (desc),11MDEF - Installazione(dta),11MDEF - Costruttore (cod),11MDEF - Anno Costruzione (cod),...,11MDEF - Numero Serie (cod),MEAS - PDR (conteggio),_key,MATRICOLA CONTATORE,PDR,MODELLO CONTATORE,VERSIONE FIRMWARE,Tecnologia di comunicazione,Data ultima comunicazione,Data ultima misura
15517,Reggio nell'Emilia,Brescello,42041,ACCESSIBILE,15441000266712,GT,515,03.12.2024,METERSIT,2024,...,MTSB030F07844927,1,MTSB030F07844927,MTSB030F07844927,15441000266712,G4_NBIOT,O732,NBIOT,04-FEB-26 11.31.30.525000000 AM,03-FEB-26 12.00.00.000000000 AM


In [27]:
#9 parsing date sulla versione pulita

def parse_oracle_date(s):
    if pd.isna(s):
        return pd.NaT
    date_part = str(s).strip().split(" ")[0]  # "16-DEC-25"
    try:
        return datetime.strptime(date_part, "%d-%b-%y")
    except ValueError:
        return pd.NaT

df_lett_re["data_lettura"] = df_lett_re[COL_DATA_LETT].apply(parse_oracle_date)

n_nat = df_lett_re["data_lettura"].isna().sum()
n_ok  = df_lett_re["data_lettura"].notna().sum()
print(f"Parsate correttamente : {n_ok:,}")
print(f"Non parsabili         : {n_nat:,}")

df_lett_re = df_lett_re.dropna(subset=["data_lettura"])

DATA_MIN = df_lett_re["data_lettura"].min()
DATA_MAX = df_lett_re["data_lettura"].max()
DURATA   = (DATA_MAX - DATA_MIN).days

print(f"Prima lettura : {DATA_MIN.date()}")
print(f"Ultima lettura: {DATA_MAX.date()}")
print(f"Durata dataset: {DURATA} giorni ({DURATA/30:.1f} mesi)")

Parsate correttamente : 298,745
Non parsabili         : 0
Prima lettura : 2025-12-06
Ultima lettura: 2025-12-22
Durata dataset: 16 giorni (0.5 mesi)


In [33]:
# Cella 10 - Target a finestre di 3 giorni

WINDOW_DAYS = 3

# Costruisce le finestre coprendo tutto il range del dataset
finestre = []
t = DATA_MIN
while t <= DATA_MAX:
    finestre.append((t, t + timedelta(days=WINDOW_DAYS - 1)))
    t += timedelta(days=WINDOW_DAYS)

print(f"Finestre costruite: {len(finestre)}")
for i, (t_start, t_end) in enumerate(finestre):
    print(f"  W{i+1}: {t_start.date()} → {t_end.date()}")

# Per ogni contatore e ogni finestra: ha comunicato?
rows = []
for key, gruppo in df_lett_re.groupby("_key"):
    date_set = set(gruppo["data_lettura"].dt.normalize())
    for i, (t_start, t_end) in enumerate(finestre):
        giorni_finestra = {t_start + timedelta(days=j) for j in range(WINDOW_DAYS)}
        comunicato = 0 if giorni_finestra & date_set else 1
        rows.append({"_key": key, "finestra": i+1, "t_start": t_start,
                     "t_end": t_end, "silente": comunicato})

# Contatori senza letture: silenti in tutte le finestre
for key in matricole_senza_lett:
    for i, (t_start, t_end) in enumerate(finestre):
        rows.append({"_key": key, "finestra": i+1, "t_start": t_start,
                     "t_end": t_end, "silente": 1})

df_target = pd.DataFrame(rows)

print(f"\nShape dataset target: {df_target.shape}")
print(f"\nDistribuzione target:")
ct = df_target["silente"].value_counts()
print(f"  Comunicato (0): {ct.get(0,0):,}  ({ct.get(0,0)/len(df_target)*100:.1f}%)")
print(f"  Silente    (1): {ct.get(1,0):,}  ({ct.get(1,0)/len(df_target)*100:.1f}%)")

print(f"\nEsempio - prime righe per un contatore:")
esempio = df_lett_re["_key"].value_counts().index[0]
display(df_target[df_target["_key"] == esempio])

Finestre costruite: 6
  W1: 2025-12-06 → 2025-12-08
  W2: 2025-12-09 → 2025-12-11
  W3: 2025-12-12 → 2025-12-14
  W4: 2025-12-15 → 2025-12-17
  W5: 2025-12-18 → 2025-12-20
  W6: 2025-12-21 → 2025-12-23

Shape dataset target: (1360212, 5)

Distribuzione target:
  Comunicato (0): 192,400  (14.1%)
  Silente    (1): 1,167,812  (85.9%)

Esempio - prime righe per un contatore:


,_key,finestra,t_start,t_end,silente
649248,MTSB036506270630,1,2025-12-06,2025-12-08,1
649249,MTSB036506270630,2,2025-12-09,2025-12-11,1
649250,MTSB036506270630,3,2025-12-12,2025-12-14,0
649251,MTSB036506270630,4,2025-12-15,2025-12-17,1
649252,MTSB036506270630,5,2025-12-18,2025-12-20,1
649253,MTSB036506270630,6,2025-12-21,2025-12-23,1
